In [ ]:
import pandas as pd
import os

selected_model = 'Simple'
base_configs = ['Ring'] #'Switch', 'FullyConnected']
options = ['slowest']# 'fastest', 'average']

def get_info(df, npu=0, perf=300, bw=2000):
    df_local = df[df["sys_id"] == npu].copy()
    compute_memory_boundary = (perf / bw) * 1e3  # FLOPS/Byte
    total_time = df_local["callback_tick"].max()
    df_memory = df_local[
        (df_local["operational_intensity"] < compute_memory_boundary)
        & (df_local["operational_intensity"] != 0)
    ]
    df_compute = df_local[df_local["operational_intensity"] >= compute_memory_boundary]
    df_idle = df_local[df_local["operational_intensity"] == 0]

    mem_time = df_memory["elapsed_time"].sum()
    comp_time = df_compute["elapsed_time"].sum()
    idle_time = df_idle["elapsed_time"].sum()
    return mem_time, comp_time, total_time - (mem_time + comp_time)

def _get_output_dir():
    base_model_output = os.path.abspath(os.path.join(os.getcwd(), "./output"))
    return base_model_output

def get_slowest_npu(df):
    max_tick = df['callback_tick'].max()
    slowest_npu = df.loc[df['callback_tick'] == max_tick, 'sys_id'].iloc[0]
    return get_info(df, npu=slowest_npu, perf=989, bw=3350)

def get_fastest_npu(df):
    max_callback = df.groupby('sys_id')['callback_tick'].max().reset_index()
    fastest_npu = max_callback.loc[max_callback['callback_tick'].idxmin(), 'sys_id']
    return get_info(df, npu=fastest_npu, perf=989, bw=3350)

def get_averaged_npus(df):
    max_npu = df["sys_id"].max() + 1
    mem_tot = 0
    comp_tot = 0
    comm_tot = 0
    for npu in range(max_npu):
        mem, comp, comm = get_info(df, npu=npu, perf=989, bw=3350)
        mem_tot += mem
        comp_tot += comp
        comm_tot += comm
    return mem_tot/max_npu, comp_tot/max_npu, comm_tot/max_npu

def get_files_list(base_dir, end_with_str):
    files = os.listdir(base_dir)
    filtered = [os.path.join(base_dir, file) for file in files if file.endswith(end_with_str)]
    return filtered

def get_all_parallelism_strategies_data(model, config, option):
    csv_files = get_files_list(os.path.join(_get_output_dir(), model, config), "_trace_matched_timing.csv")
    records = []
    for file in csv_files:
        df = pd.read_csv(file)
        df.fillna(0, inplace=True)
        if option == 'slowest':
            mem, comp, comm = get_slowest_npu(df)
        elif option == 'average':
            mem, comp, comm = get_averaged_npus(df)
        elif option == 'fastest':
            mem, comp, comm = get_fastest_npu(df)
        else:
            return []
        base_name = os.path.basename(file)
        clean_name = base_name.split('_trace')[0]
        dp, tp, sp, pp, fsdp = clean_name.split('_')
        total = mem + comp + comm
        records.append({
            'file_name': clean_name,
            'dp': int(dp),
            'tp': int(tp),
            'sp': int(sp),
            'pp': int(pp),
            'fsdp': int(fsdp),
            'mem': mem,
            'comp': comp,
            'comm': comm,
            'mem_percent': mem/total*100,
            'comp_percent': comp/total*100,
            'comm_percent': comm/total*100,
            'total': total,
            'config_tuple': (int(dp), int(tp), int(sp), int(pp), int(fsdp))
        })
    return pd.DataFrame(records)

def get_order(df):
    # Return list of config_tuples sorted by total
    return df.sort_values('total')['config_tuple'].tolist()

def compare_orders(df_unaware, df_aware):
    unaware_order = get_order(df_unaware)
    aware_order = get_order(df_aware)
    unaware_rank = {cfg: i for i, cfg in enumerate(unaware_order)}
    aware_rank = {cfg: i for i, cfg in enumerate(aware_order)}
    # Find configs present in both
    common = set(unaware_order) & set(aware_order)
    diffs = [abs(unaware_rank[cfg] - aware_rank[cfg]) for cfg in common]
    only_unaware = set(unaware_order) - set(aware_order)
    only_aware = set(aware_order) - set(unaware_order)
    return {
        'sum_order_diff': sum(diffs),
        'n_matched': len(diffs),
        'n_only_unaware': len(only_unaware),
        'n_only_aware': len(only_aware),
        'matched_diffs': diffs,
        'matched_configs': list(common),
        'only_unaware': list(only_unaware),
        'only_aware': list(only_aware),
    }

results = []
detailed_rows = []

for option in options:
    for base_cfg in base_configs:
        unaware_cfg = base_cfg
        aware_cfg = base_cfg + '_Unaware'
        try:
            df_unaware = get_all_parallelism_strategies_data(selected_model, unaware_cfg, option)
            df_aware = get_all_parallelism_strategies_data(selected_model, aware_cfg, option)
        except Exception as e:
            print(e)
            continue

        # --- Order comparison (preserved from before) ---
        def get_order(df):
            return df.sort_values('total')['config_tuple'].tolist()
        unaware_order = get_order(df_unaware)
        aware_order = get_order(df_aware)
        unaware_rank = {cfg: i for i, cfg in enumerate(unaware_order)}
        aware_rank = {cfg: i for i, cfg in enumerate(aware_order)}
        common = set(unaware_order) & set(aware_order)
        diffs = [abs(unaware_rank[cfg] - aware_rank[cfg]) for cfg in common]
        only_unaware = set(unaware_order) - set(aware_order)
        only_aware = set(aware_order) - set(unaware_order)
        cmp = {
            'sum_order_diff': sum(diffs),
            'n_matched': len(diffs),
            'n_only_unaware': len(only_unaware),
            'n_only_aware': len(only_aware),
            'matched_diffs': diffs,
            'matched_configs': list(common),
            'only_unaware': list(only_unaware),
            'only_aware': list(only_aware),
            'option': option,
            'config': base_cfg
        }
        results.append(cmp)

        # --- Detailed mem/comp/comm comparison for matched configs ---
        df_unaware = df_unaware.set_index('config_tuple')
        df_aware = df_aware.set_index('config_tuple')
        for cfg_tuple in common:
            row_unaware = df_unaware.loc[[cfg_tuple]]
            row_aware = df_aware.loc[[cfg_tuple]]
            for key in ['mem', 'comp', 'comm']:
                abs_diff = (row_aware[key] - row_unaware[key]).values[0]
                diff = (abs_diff / row_unaware[key]).values[0]
                rel_diff = diff if row_unaware[key].values[0] != 0 else float('nan')

                detailed_rows.append({
                    'option': option,
                    'config': base_cfg,
                    'config_tuple': cfg_tuple,
                    'type': key,
                    'unaware': row_unaware[key].values[0],
                    'aware': row_aware[key].values[0],
                    'abs_diff': abs_diff,
                    'rel_diff': rel_diff
                })

# Summary order comparison
summary_df = pd.DataFrame(results)
display(summary_df)

# Detailed mem/comp/comm comparison
detailed_df = pd.DataFrame(detailed_rows)
display(detailed_df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Filter only mem/comp/comm rows
plot_df = detailed_df[detailed_df['type'].isin(['mem', 'comp', 'comm'])].copy()

# Pivot to get columns: ['unaware_mem', 'unaware_comp', ...]
pivot_unaware = plot_df.pivot_table(index='config_tuple', columns='type', values='unaware', aggfunc='first')
pivot_aware = plot_df.pivot_table(index='config_tuple', columns='type', values='aware', aggfunc='first')

# Compute total time for sorting
pivot_unaware['total'] = pivot_unaware[['mem', 'comp', 'comm']].sum(axis=1)
pivot_aware['total'] = pivot_aware[['mem', 'comp', 'comm']].sum(axis=1)

# Sort by unaware total
print(pivot_unaware)
sorted_idx = pivot_unaware.sort_values('total').index

# Prepare data for plotting
bar_width = 0.35
x = np.arange(len(sorted_idx))

fig, ax = plt.subplots(figsize=(14, max(6, len(sorted_idx)//2)))

# Plot unaware bars
bottom_unaware = np.zeros(len(sorted_idx))
for part, color in zip(['mem', 'comp', 'comm'], ['#1f77b4', '#ff7f0e', '#2ca02c']):
    vals = pivot_unaware.loc[sorted_idx, part].values
    ax.bar(x - bar_width/2, vals, bar_width, bottom=bottom_unaware, label=f'Unaware {part}', color=color, alpha=0.7)
    bottom_unaware += vals

# Plot aware bars
bottom_aware = np.zeros(len(sorted_idx))
for part, color in zip(['mem', 'comp', 'comm'], ['#1f77b4', '#ff7f0e', '#2ca02c']):
    vals = pivot_aware.loc[sorted_idx, part].values
    ax.bar(x + bar_width/2, vals, bar_width, bottom=bottom_aware, label=f'Aware {part}', color=color, alpha=0.35, hatch='//')
    bottom_aware += vals

# X labels
ax.set_xticks(x)
ax.set_xticklabels([str(t) for t in sorted_idx], rotation=90)
ax.set_xlabel('Config (dp, tp, sp, pp, fsdp)')
ax.set_ylabel('Time')
ax.set_title('Stacked mem/comp/comm bars for Unaware (solid) and Aware (hatched), sorted by Unaware total time')
# Custom legend
handles = [
    plt.Rectangle((0,0),1,1, color='#1f77b4', alpha=0.7, label='Unaware mem'),
    plt.Rectangle((0,0),1,1, color='#ff7f0e', alpha=0.7, label='Unaware comp'),
    plt.Rectangle((0,0),1,1, color='#2ca02c', alpha=0.7, label='Unaware comm'),
    plt.Rectangle((0,0),1,1, color='#1f77b4', alpha=0.35, hatch='//', label='Aware mem'),
    plt.Rectangle((0,0),1,1, color='#ff7f0e', alpha=0.35, hatch='//', label='Aware comp'),
    plt.Rectangle((0,0),1,1, color='#2ca02c', alpha=0.35, hatch='//', label='Aware comm'),
]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Use the same sorting as before
sorted_idx = pivot_unaware.sort_values('total').index
x = np.arange(len(sorted_idx))
bar_width = 0.35

for part, color in zip(['mem', 'comp', 'comm'], ['#1f77b4', '#ff7f0e', '#2ca02c']):
    fig, ax = plt.subplots(figsize=(16, 6))
    unaware_vals = pivot_unaware.loc[sorted_idx, part].values
    aware_vals = pivot_aware.loc[sorted_idx, part].values

    ax.bar(x - bar_width/2, unaware_vals, bar_width, label='Unaware', color=color, alpha=0.7)
    ax.bar(x + bar_width/2, aware_vals, bar_width, label='Aware', color=color, alpha=0.35, hatch='//')

    ax.set_xticks(x)
    ax.set_xticklabels([str(t) for t in sorted_idx], rotation=90)
    ax.set_xlabel('Config (dp, tp, sp, pp, fsdp)')
    ax.set_ylabel(f'{part} time')
    ax.set_title(f'{part.capitalize()} time: Unaware vs Aware (sorted by Unaware total time)')
    ax.legend()
    plt.tight_layout()
    plt.show()